## Harness Testing for MiniMax M 2.1 within langchain

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="unsloth/Nemotron-3-Super",   
    api_key="fake_key_for_llama_cpp",
    base_url="http://localhost:8090/v1",         
)

set up Wikipedia tools

In [ ]:
import asyncio

from wikipedia_tool.main import WikipediaToolKit

USER_AGENT = "TriviaQA (dzureca7@gmail.com)"

wikipedia = WikipediaToolKit(USER_AGENT)
tools = [wikipedia.search, wikipedia.inspect]

Create the langchain agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, tools)

Extract tool calls

In [ ]:
def extract_tool_names(result: dict) -> list[str]:
    tool_names = set()
    messages = result.get("messages", [])
    
    for msg in messages:
        # Check if the message has the tool_calls attribute (typical for AIMessage)
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for call in msg.tool_calls:
                tool_names.add(call["name"])
                
    return sorted(list(tool_names))

Extract Final Answer

In [ ]:
def get_final_answer(result: dict) -> str:
    messages = result.get("messages", [])
    # Search backwards for the last AI message with actual text content
    for msg in reversed(messages):
        # In LangChain, messages are objects, so we access .type and .content
        if msg.type == "ai" and msg.content:
            return msg.content
    return "No answer found."

Test run the agent

In [ ]:
async def run_agent(question: str):
    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
    # Optional: print raw result for debugging
    # print(result)
    
    raw_answer = get_final_answer(result)
    tools_used = extract_tool_names(result)
    
    return (tools_used, raw_answer)

Invocation

In [ ]:
tools, answer = await run_agent("Search Wikipedia and tell me a short summary of the history of Nvidia Corporation.")
print("Tools used ➡️", tools)
print(answer)